# M2 canonical acquisition
Thin, restart-safe orchestration of reviewed repository code. Gate B remains blocked unless the reviewed target evidence is `confirmed`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
WORKSPACE='/content/drive/MyDrive/giab-wes-nextflow-private'
STAGING='/content/m2-stage'
REPO='/content/giab-wes-nextflow'
REF='feat/m2-data-provenance'  # replace with reviewed commit SHA
RUN_ID='m2-20260903'
TARGET_BED=None  # set only after the reviewed capture-design gate is confirmed
SOURCE_DICT=None  # matching hg19/GRCh37 Picard sequence dictionary
from pathlib import Path
assert Path(WORKSPACE).name == 'giab-wes-nextflow-private'
assert not (Path(WORKSPACE)/'DO NOT ACCESS WITH CHATGPT').exists(), 'Safety marker present'


In [ ]:
import shutil, subprocess
if not Path(REPO).exists():
    subprocess.run(['git','clone','https://github.com/jcollins-bioinfo/giab-wes-nextflow.git',REPO],check=True)
subprocess.run(['git','-C',REPO,'fetch','--all','--prune'],check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',REF],check=True)
repo_sha=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
print('repository SHA:',repo_sha)


In [ ]:
import json, os, platform
for path in ['/content',WORKSPACE]:
    usage=shutil.disk_usage(path); print(path,'free_GiB',round(usage.free/2**30,1))
assert shutil.disk_usage('/content').free >= 100*2**30, 'Need >=100 GiB ephemeral space'
runtime={'repository_sha':repo_sha,'platform':platform.platform(),'architecture':platform.machine(),'python':platform.python_version()}
Path(STAGING).mkdir(parents=True,exist_ok=True)
(Path(STAGING)/'runtime.json').write_text(json.dumps(runtime,sort_keys=True,indent=2)+'\n')
print(runtime)


In [ ]:
subprocess.run(['python','-m','pip','install','-r',f'{REPO}/requirements-dev.txt'],check=True)
# The repository driver processes manifest objects sequentially and is safe to rerun.
subprocess.run(['python',f'{REPO}/scripts/acquire_m2.py','--workspace',STAGING,'--run-id',RUN_ID],check=True)


In [ ]:
gate=json.loads(Path(f'{REPO}/config/m2-target-design.json').read_text())
prepare=['python',f'{REPO}/scripts/prepare_m2.py','--workspace',STAGING,'--run-id',RUN_ID]
if gate['classification']=='confirmed':
    if not TARGET_BED or not SOURCE_DICT:
        raise RuntimeError('Set reviewed TARGET_BED and SOURCE_DICT paths; hidden state is forbidden.')
    prepare.extend(['--target-bed',TARGET_BED,'--source-dict',SOURCE_DICT])
    subprocess.run(prepare,check=True)
    subprocess.run(['python',f'{REPO}/scripts/validate_m2.py','--workspace',STAGING,'--run-id',RUN_ID],check=True)
    subprocess.run(['python',f'{REPO}/scripts/publish_m2_workspace.py','--staging',STAGING,'--drive-root',WORKSPACE,'--run-id',RUN_ID],check=True)
    print('Gate B publication completed:',RUN_ID)
else:
    # Reference preparation is allowed, but canonical liftover/domain publication is not.
    subprocess.run(prepare,check=True)
    print('Gate A ready; Gate B BLOCKED:',gate['block_reason'])
    print('Recovery: confirm exact capture-design identity, review the gate change, set TARGET_BED/SOURCE_DICT, and rerun this cell.')
